So far we've been doing a process called *analog* Monte Carlo. We can change the way the Monte Carlo "game" is played and adjust our sampling to be more favorable. 


<div class="alert alert-block alert-info">
<h2> <b>Think-pair-share:</b> Think about how you might change a problem formulation to get particles to a space in your simulation that has low sampling... What might you do? What considerations should one make?</h2>
</div>

*Implicit Capture* is a method where we modify our sampling in a problem where capture is important. We modify our sampling such that particle weights are adjusted as a particle transits the problem. We don't sample every capture reaction, but instead reduce a particle weight to account for capture reactions. 

The algorithm for implicit capture is:
* Create a counter, t = 0 to track the number of neutrons that get through.
* Create a neutron with $\mu$ sampled from the uniform distribution in [0,1]. Set x=0. Set the particle's weight to be $w=1/N$
* Sample randomly a distance to scatter, $l$, from the exponential distribution.
* Move the particle to $x = x + l\mu$.
* Reduce the weight of the particle by a factor $e^{-\Sigma_\gamma s}$.
* Check to see if x > 3. If so, go to t = t+w, and go to 2. Otherwise, if x < 0 to to step 2.
* Go back to step 3. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def slab_transmission(Sig_s,Sig_a,thickness,N,isotropic=False, implicit_capture = True):
    """Compute the fraction of neutrons that leak through a slab
    Inputs:
    Sig_s:     The scattering macroscopic x-section
    Sig_a:     The absorption macroscopic x-section
    thickness: Width of the slab
    N:         Number of neutrons to simulate
    isotropic: Are the neutrons isotropic or a beam
    
    Returns:
    transmission:  The fraction of neutrons that made it through
    """
    Sig_t = Sig_a + Sig_s
    transmission = 0.0
    N = int(N)
    for i in range(N):
        if (isotropic):
            mu = np.random.random(1)
        else:
            mu = 1.0
        x = 0
        alive = 1
        weight = 1.0/N
        while (alive):
            if (implicit_capture):
                #get distance to collision
                if (Sig_s > 0):
                    l = -np.log(1-np.random.random(1))/Sig_s 
                else:
                    l = 10.0*thickness/mu #something that will make it through
            else:
                #get distance to collision
                l = -np.log(1-np.random.random(1))/Sig_t
            #make sure that l is not too large. If it is, move it to the edge.
            if (mu > 0):
                val = (thickness-x)/mu
                l = np.min(np.array([l,val], dtype=object)) 
            else:
                l = np.min([l,-x/mu])
            #move particle
            x += l*mu
            if (implicit_capture):
                if not(l>=0):
                    print(l,x,mu)
                assert(l>=0)
                weight *= np.exp(-l*Sig_a)
            #still in the slab? 
            #It should be either at the edge of the slab on the right, or have a negative x value
            if (np.abs(x-thickness) < 1.0e-14):
                transmission += weight
                alive = 0
            elif (x<= 1.0e-14):
                alive = 0
            else:
                if (implicit_capture):
                    mu = np.random.uniform(-1,1,1)
                else:
                    #scatter or absorb
                    if (np.random.random(1) < Sig_s/Sig_t): 
                        #scatter, pick new mu
                        mu = np.random.uniform(-1,1,1)
                    else: #absorbed
                        alive = 0
    return transmission



Now let's try a problem where there's no scattering and only absorption. Implicit capture works well in problems where capture is important; a problem with no scattering is an extreme case of this. 

With a pure absorber, we can calculate the exact answer for transmission. Let's see the difference with implicit capture and analog Monte Carlo using one particle in the simulation. 

In [ ]:
N = 1
Sigma_s = 0.0
Sigma_a = 2.0
thickness = 3
transmission = slab_transmission(Sigma_s,Sigma_a, thickness,
                                 N, isotropic=False, implicit_capture=True)
print("The fraction that made it through using implicit capture was", transmission, "with a percent error of",
      np.abs(transmission - np.exp(-6))/np.exp(-6)*100,"%")
transmission = slab_transmission(Sigma_s,Sigma_a, thickness, 
                                 N, isotropic=False, implicit_capture=False)
print(transmission)
print("The fraction that made it through using analog tracking was", transmission, "with a percent error of", 
      np.abs(transmission - np.exp(-6))/np.exp(-6)*100,"%")

With an isotropic particle distribution in the same problem, we won't get the exact answer with a single particle, but we can expect it to perform better in a problem with the same material properties and dimensions. Let's try with 10,000 particles and compare them... 

In [ ]:
true_sol = 0.00031825746369040646727
N=10000
transmission = slab_transmission(Sigma_s,Sigma_a, thickness, N, isotropic=True, implicit_capture=True)
print("The fraction that made it through using implicit capture was", transmission, "with a percent error of",np.abs(transmission - true_sol)/true_sol*100,"%")
transmission = slab_transmission(Sigma_s,Sigma_a, thickness,
                                 N, isotropic=True, implicit_capture=False)
print("The fraction that made it through using analog tracking was",
      transmission, "with a percent error of",
      np.abs(transmission - true_sol)/true_sol*100,"%")

Variance reduction methods, like implicit capture, reduce the time or number of particles needed to get to a particular solution (with a desired uncertainty) by changing the "game" that is being played to simulate these particles. As we discussed in implicit capture, because we changed how we sample distance to collision, we had to modify the particle weight accordingly and consistently. 

Many variance reduction methods exist. Two additional methods are russian roulette and splitting. In russian roulette, we systematically get rid of particles and increase surviving particle weights so that the total weight of the problem remains unchanged. 

In splitting we increase the number of particles and decrease their weights to the total weight of the problem remains unchanged. 


<div class="alert alert-block alert-info">
<h2> <b>Think-pair-share:</b> When would you want to use rouletting? When would you want to use splitting?</h2>
</div>

In [ ]:

def russian_roulette(weight, wa):
    """Perform Russian roulette on Neutron
    Inputs:
    weight:        current weight
    wa:            average weight of surviving neutrons
    Returns:
    alive:         0,1 for whether neutron survived
    weight_final:  weight after roulette
    """
    pk = 1- weight/wa
    alive = np.random.random(1)<pk
    weight_final = wa*(alive) + 0
    return alive, weight_final

In [ ]:
def split(neutron, wd):
    """Perform Russian roulette on Neutron
    Inputs:
    neutron:       census entry for current neutron
    wd:            desired weight for split neutrons
    Returns:
    new_neutrons:  census entries for new neutrons
    """
    w = neutron[0]
    num_split = int(np.round(w/wd))
    wsplit = w/num_split
    new_neutrons = np.zeros([num_split-1,7])
    neutron[0] = wsplit
    for i in range(num_split-1):
        new_neutrons[i,:] = neutron.copy()
    return new_neutrons

So far we've been focusing on how to calculate transmissions through slabs, but we may want to consider other behaviors in our problems. 



<div class="alert alert-block alert-info">
<h2> <b>Think-pair-share:</b> What sorts of quantities would you want to estimate with Monte Carlo? How do you think you might approach determining these with monte Carlo?</h2>
</div>



Our transmission calculations so far have involved particles crossing a surface, but let's consider how we might quantify material happening within a volume, or cell. The neutron flux can be estimated in two ways: through collision estimators and track length estimators. 

In [ ]:
def slab_source(Nx,Sig_s,Sig_a,thickness,N,Q,isotropic=False, implicit_capture = True):
    """Compute the fraction of neutrons that leak through a slab
    Inputs:
    Nx:        The number of grid points
    Sig_s:     The scattering macroscopic x-section
    Sig_a:     The absorption macroscopic x-section
    thickness: Width of the slab
    N:         Number of neutrons to simulate
    Q:         Source strength
    isotropic: Are the neutrons isotropic or a beam
    
    Returns:
    transmission:  The fraction of neutrons that made it through
    scalar_flux:   The scalar flux in each of the Nx cells
    X:             The value of the cell centers in the mesh
    """
    dx = thickness/Nx
    X = np.linspace(dx*0.5, thickness - 0.5*dx,Nx)
    scalar_flux = np.zeros(Nx)
    Sig_t = Sig_a + Sig_s
    leak_left = 0.0
    leak_right = 0
    N = int(N)
    for i in range(N):
        if (isotropic):
            mu = np.random.uniform(-1,1,1)
        else:
            mu = 1.0
        x = np.random.random(1)*thickness
        alive = 1
        weight = Q*thickness/N
        while (alive):
            if (implicit_capture):
                #get distance to collision
                if (Sig_s > 0):
                    l = -np.log(1-np.random.random(1))/Sig_s 
                else:
                    l = 10.0*thickness/mu #something that will make it through
            else:
                #get distance to collision
                l = -np.log(1-np.random.random(1))/Sig_t
            #make sure that l is not too large
            if (mu > 0):
                l = np.min([l,(3-x)/mu]) 
            else:
                l = np.min([l,-x/mu])
            #move particle
            x += l*mu
            if (implicit_capture):
                if not(l>=0):
                    print(l,x,mu)
                assert(l>=0)
                weight_old = weight
                weight *= np.exp(-l*Sig_a)
            #still in the slab?
            if (np.abs(x-thickness) < 1.0e-14):
                leak_right += weight
                alive = 0
            elif (x<= 1.0e-14):
                alive = 0
                leak_left += weight
            else:
                #compute cell particle collision is in
                cell= np.argmin(np.abs(X-x))
                if (implicit_capture):
                    mu = np.random.uniform(-1,1,1)
                    scalar_flux[cell] += weight/Sig_s/dx
                else:
                    #scatter or absorb
                    scalar_flux[cell] += weight/Sig_t/dx
                    if (np.random.random(1) < Sig_s/Sig_t): 
                        #scatter, pick new mu
                        mu = np.random.uniform(-1,1,1)
                    else: #absorbed
                        alive = 0
    return leak_left,leak_right, scalar_flux, X

Similarly, we can construct neutrons transmitting through a slab using a track length estimator for the flux. Here we'll get both... 

In [ ]:
def slab_source2(Nx,Sig_s,Sig_a,thickness,N,Q,isotropic=False, implicit_capture = True):
    """Compute the fraction of neutrons that leak through a slab
    Inputs:
    Nx:        The number of grid points
    Sig_s:     The scattering macroscopic x-section
    Sig_a:     The absorption macroscopic x-section
    thickness: Width of the slab
    N:         Number of neutrons to simulate
    isotropic: Are the neutrons isotropic or a beam
    
    Returns:
    transmission:  The fraction of neutrons that made it through
    scalar_flux:   The scalar flux in each of the Nx cells
    scalar_flux_tl:   The scalar flux in each of the Nx cells from track length estimator
    X:             The value of the cell centers in the mesh
    """
    dx = thickness/Nx
    X = np.linspace(dx*0.5, thickness - 0.5*dx,Nx)
    scalar_flux = np.zeros(Nx)
    scalar_flux_tl = np.zeros(Nx)
    Sig_t = Sig_a + Sig_s
    leak_left = 0.0
    leak_right = 0
    N = int(N)
    for i in range(N):
        if (isotropic):
            mu = np.random.uniform(-1,1,1)
        else:
            mu = 1.0
        x = np.random.random(1)*thickness
        alive = 1
        weight = Q*thickness/N
        #which cell am I in
        cell = np.argmin(np.abs(X-x))
        while (alive):
            if (implicit_capture):
                #get distance to collision
                if (Sig_s > 0):
                    l = -np.log(1-np.random.random(1))/Sig_s 
                else:
                    l = 10.0*thickness/np.abs(mu) #something that will make it through
            else:
                #get distance to collision
                l = -np.log(1-np.random.random(1))/Sig_t
            #compare distance to collision to distance to cell edge
            distance_to_edge = ((mu > 0.0)*( (cell+1)*dx - x) + 
                                (mu<0.0)*( x - cell*dx) + 1.0e-8)/np.abs(mu)
            if (distance_to_edge < l):
                l = distance_to_edge
                collide = 0
            else:
                collide = 1
            #move particle
            x += l*mu
            #score track length tally
            if (implicit_capture):
                scalar_flux_tl[cell] += weight*(1.0 - np.exp(-l*Sig_a))/(Sig_a + 1.0e-14)
            else:
                scalar_flux_tl[cell] += weight*l
            if (implicit_capture):
                if not(l>=0):
                    print(l,x,mu,cell,distance_to_edge)
                assert(l>=0)
                weight_old = weight
                weight *= np.exp(-l*Sig_a)
            #still in the slab?
            if (np.abs(x-thickness) < 1.0e-14) or (x > thickness):
                leak_right += weight
                alive = 0
            elif (x<= 1.0e-14):
                alive = 0
                leak_left += weight
            else:
                #compute cell particle collision is in
                cell= np.argmin(np.abs(X-x))
                if (implicit_capture):
                    if (collide):
                        mu = np.random.uniform(-1,1,1)
                    scalar_flux[cell] += weight/Sig_s/dx
                else:
                    #scatter or absorb
                    scalar_flux[cell] += weight/Sig_t/dx
                    if (collide) and (np.random.random(1) < Sig_s/Sig_t): 
                        #scatter, pick new mu
                        mu = np.random.uniform(-1,1,1)
                    elif (collide): #absorbed
                        alive = 0
            #print(x,mu,alive,l*mu,weight*l)
    return leak_left,leak_right, scalar_flux, scalar_flux_tl/dx, X


So far we've been using random number generators over whole spans of a domain. We can use a method called stratified sampling to force particles to be evenly distributed over a particular domian. Imagine, for example, that you bin your domain in 10 regions and force each region to have the same number of samples born into it each one. This is the principle used in stratified sampling. 

Imagine we take N number of samples, and want to split those N particles in S bins (the number of strata). N must be divisible by S so that it can be evenly distributed across all S strata. 

Below is code to sample given equally spaced strata between 0 and 1: 

In [ ]:
def strat_sample(N,S):    
    """Create N samples in S strata.
    N must be divisible by S
    Inputs:
    N:             number of samples
    S:             number of strata
    Returns:
    place_in_bin:  a numpy vector containing the samples
    """
    N = N + (N % S)
    assert(N%S == 0 )
    dS = 1.0/S
    bins = np.zeros(N,dtype=int)
    count = 0
    for i in range(N//S):
        bins[count:count+S] = np.random.permutation(S)
        count += S
    place_in_bin = np.random.uniform(-0.5*dS,0.5*dS,N) + (bins+0.5)*dS
    return place_in_bin

This can be extended into two dimensions (SxS strata). 

In [ ]:
def strat_sample_2D(N,S): 
    """Create N samples in S*S strata.
    Inputs:
    N:             number of samples
    S:             number of strata in each dimension
    Returns:
    samples:       an N by 2 numpy vector containing the samples
    """
    #number of bins is S*S
    bins = S*S
    #make sure we have enough points
    if (N<bins):
        N = bins
    N += N % bins
    Num_per_bin = N//bins
    assert(N % bins == 0)
    samples = np.zeros((N,2))
    count = 0;
    for bin_x in range(S):
        for bin_y in range(S):
            for i in range(Num_per_bin):
                center = (bin_x/S + 0.5/S,bin_y/S + 0.5/S)
                samples[count,0:2] = center + np.random.uniform(low=-0.5,high=0.5,size=2)/S
                count += 1
            
    return samples

Bringing this all together, below is a slab function that can use implicit capture, tallies the flux with both collision and track length estimators, and with stratified sampling: 

In [ ]:
def slab_source(Nx,Sig_s,Sig_a,thickness,a,b,N,Q,
                implicit_capture = True, cutoff = 1.0e-3, stratified = [1,1]):
    """Compute the fraction of neutrons that leak through a slab
    Inputs:
    Nx: The number of grid points
    Sig_s: The scattering macroscopic x-section
    Sig_a: The absorption macroscopic x-section
    thickness: Width of the slab
    a,b: Endpoints of Source
    N: Number of neutrons to simulate
    implicit_capture: Do we run implicit capture
    cutoff: At what level do we stop implicit capture
    stratified: Use stratified sampling in space and angle
                Specify a list of length two with the number of
                strata in each dimension; default [1,1] for unstratified
    Returns:
    transmission: The fraction of neutrons that made it through
    scalar_flux: The scalar flux in each of the Nx cells
    scalar_flux_tl: The scalar flux in each of the Nx cells
                    from track length estimator
    X: The value of the cell centers in the mesh
    """
    imp_input = implicit_capture
    dx = thickness/Nx
    X = np.linspace(dx*0.5, thickness - 0.5*dx,Nx)
    scalar_flux = np.zeros(Nx)
    scalar_flux_tl = np.zeros(Nx)
    assert (Sig_s.size == Nx) and (Sig_a.size == Nx)
    Sig_t = Sig_a + Sig_s
    iSig_t = 1.0/Sig_t
    iSig_s = 1.0/(Sig_s+1.0e-14)
    iSig_a = 1.0/(Sig_a+1.0e-14)
    leak_left = 0.0
    leak_right = 0
    N = int(N)
    #make a vector of the initial positions and mus
    samples = strat_sample_2D(N,stratified[0],stratified[1])
    xs = samples[:,0]*(b-a) + a #adjust to bounds of source
    mus = (samples[:,1]-0.5)*2 #shift to range -1 to 1
    N = int(xs.size)
    #the initial weight does not change
    init_weight = Q*thickness/N
    for i in range(N):
        mu = mus[i]
    x = xs[i]
    alive = 1
    weight = init_weight
    #which cell am I in
    cell = int(x/dx)
    implicit_capture = imp_input
    while (alive):
        if (weight < cutoff*init_weight):
            implicit_capture = False
        if (implicit_capture):
            l = -math.log(1-random.random())*iSig_s[cell]
        else:
            #get distance to collision
            l = -math.log(1-random.random())*iSig_t[cell]
        #compare distance to collision to distance to cell edge
        distance_to_edge = ((mu > 0.0)*( (cell+1)*dx - x) +
                            (mu<0.0)*( x - cell*dx) + 1.0e-8)/math.fabs(mu)
        if (distance_to_edge < l):
            l = distance_to_edge
            collide = 0
        else:
            collide = 1
        x += l*mu #move particle
        #score track length tally
        if (implicit_capture):
            scalar_flux_tl[cell] += weight*(1.0 -
                                            math.exp(-l*Sig_a[cell]))*iSig_a[cell]
        else:
            scalar_flux_tl[cell] += weight*l
        if (implicit_capture):
            weight *= math.exp(-l*Sig_a[cell])
        #still in the slab?
        if (math.fabs(x-thickness) < 1.0e-14) or (x > thickness):
            leak_right += weight
            alive = 0
        elif (x<= 1.0e-14):
            alive = 0
            leak_left += weight
        else:
            cell= int(x/dx) #compute cell particle collision is in
            if (implicit_capture):
                if (collide):
                    mu = random.uniform(-1,1)
                scalar_flux[cell] += weight*iSig_s[cell]/dx
            else: #scatter or absorb
                scalar_flux[cell] += weight*iSig_t[cell]/dx
                if (collide) and (random.random() < Sig_s[cell]*iSig_t[cell]):
                    #scatter, pick new mu
                    mu = random.uniform(-1,1)
                elif (collide): #absorbed
                    alive = 0
    return leak_left,leak_right, scalar_flux, scalar_flux_tl/dx, X, N

For your final project, there are a number of approaches that you might take to create your solver. Here are some helpful functions that might be useful as starting points for you in doing your project. 

In [ ]:
def create_particles(N,Q,X,Y,dx,dy):
    """Create N source particles in 2-D regular grid with source strengths in the 2-D array Q
    Inputs:
    N:         Number of neutrons to create
    Q:         2-D array of source strengths
    X,Y:       2-D array of zone centers
    dx,dy:     Width and height of zones
    
    Returns:
    census:    N by 7 array containing, weight, position (x,y), mu, gamma, and zone numbers
    """
    total = np.sum(Q)
    I,J = Q.shape
    census = np.empty((1,7))
    for i in range(I):
        for j in range(J):
            if Q[i,j] > 1.0e-14:
                num_emit = (np.ceil(Q[i,j]/total*N))
                #set weight
                wgt = Q[i,j]*dx*dy/(num_emit+1.0e-14)
                for emit in range(int(num_emit)):
                    #set position
                    pos = np.random.uniform(-0.5,0.5,2)
                    x_pos = dx * pos[0] + X[i,j]
                    y_pos = dy * pos[1] + Y[i,j]
                    mu = np.random.uniform(-1,1,1)
                    gamma = np.random.uniform(0,2*np.pi,1)
                    census = np.vstack((census,[wgt,x_pos,y_pos,mu[0],gamma[0],i,j]))
    return np.delete(census,0,axis=0)

def move_particles(census,X,Y,dx,dy,Sig_t,Sig_a,implicit_capture = True):
    """Create N source particles in 2-D regular grid with source strengths in the 2-D array Q
    Inputs:
    census:    List of particles created by the source function
    X,Y:       2-D arrays of cell centers
    dx,dy:     Widths of zones
    Sig_t:     2-D array of total macroscopic cross-sections
    Sig_a:     2-D array of absorption macroscopic cross-sections
    implicit_capture: whether or not to use implicit capture tracking
    
    Returns:
    scalar_flux_coll:    collision-estimated scalar flux array the same size as X and Y
    scalar_flux_tl:      track-length-estimated scalar flux array the same size as X and Y
    """
    Sig_s = Sig_t - Sig_a
    scalar_flux_coll = 0*X + 1e-14
    scalar_flux_tl =  0*X + 1e-14
    Lx, Ly = X.shape
    for neut in census:
        alive = 1
        while (alive):
            cell = np.array(neut[5:7], dtype=int)
            #compute distance to collision
            if (implicit_capture):
                #get distance to collision
                l = -np.log(1-np.random.random(1))/(Sig_s[cell[0], cell[1]] + 1.0e-14)
            else:
                #get distance to collision
                l = -np.log(1-np.random.random(1))/Sig_t[cell[0], cell[1]]
            #distance to x boundary
            center = [ X[cell[0], cell[1]], Y[cell[0], cell[1]]]
            pos = neut[1:3]
            mu = neut[3]
            gamma = neut[4]
            omega_x = np.sqrt(1.0-mu*mu)*np.cos(gamma)
            omega_y = np.sqrt(1.0-mu*mu)*np.sin(gamma)
            if (omega_x > 0):
                dist_x = (center[0] + dx*0.5 - pos[0])/omega_x + 1.0e-14
            else:
                dist_x = -(pos[0] - (center[0] - dx*0.5))/omega_x + 1.0e-14
            if (omega_y > 0):
                dist_y = (center[1] + dy*0.5 - pos[1])/omega_y + 1.0e-14
            else:
                dist_y = -(pos[1] - (center[1] - dy*0.5))/omega_y + 1.0e-14
            assert(dist_y>0)
            assert(dist_x>0)
            
            #find smallest distance
            if (l < dist_x) and (l < dist_y):
                neut[1] += l*omega_x
                neut[2] += l*omega_y
                #score in collision tally
                if (implicit_capture):
                    scalar_flux_coll[cell[0], cell[1]] += neut[0]/Sig_s[cell[0], cell[1]]
                else:
                    scalar_flux_coll[cell[0], cell[1]] += neut[0]/Sig_t[cell[0], cell[1]]
                if (implicit_capture and (Sig_a[cell[0], cell[1]] > 0)):
                    scalar_flux_tl[cell[0],cell[1]] += neut[0]*((1.0 - 
                                                                 np.exp(-l*Sig_a[cell[0], cell[1]]))
                                                                /(Sig_a[cell[0], cell[1]] + 1.0e-14))
                else:
                    scalar_flux_tl[cell[0],cell[1]] += neut[0]*l
                
                if (implicit_capture):
                    neut[3] = np.random.uniform(-1,1,1)
                    neut[4] = np.random.uniform(0,2*np.pi,1)
                    neut[0] *= np.exp(-l*Sig_a[cell[0], cell[1]] )
                else:
                    #scatter or absorb
                    if (np.random.random(1) < Sig_s[cell[0], cell[1]]/Sig_t[cell[0], cell[1]]): 
                        #scatter, pick new mu
                        neut[3] = np.random.uniform(-1,1,1)
                        neut[4] = np.random.uniform(0,2*np.pi,1)
                    else: #absorbed
                        #print("killed")
                        alive = 0
            elif (l >= dist_x) or (l >= dist_y):
                if (dist_y < dist_x):
                    pos[0] += (dist_y)*omega_x
                    neut[6] += np.sign(omega_y)
                    pos[1] += (dist_y + 1e-10)*omega_y
                    neut[1] = pos[0]
                    neut[2] = pos[1]
                    if (implicit_capture) and (Sig_a[cell[0], cell[1]] > 0):
                        scalar_flux_tl[cell[0],cell[1]] += neut[0]*((1.0 - 
                                                                     np.exp(-dist_y*Sig_a[cell[0], cell[1]]))
                                                                    /(Sig_a[cell[0], cell[1]] + 1.0e-14))
                        neut[0] *= np.exp(-dist_y*Sig_a[cell[0], cell[1]] )
                    else:
                        scalar_flux_tl[cell[0],cell[1]] += neut[0]*dist_y
                else:
                    pos[1] += (dist_x)*omega_y
                    neut[5] += np.sign(omega_x)
                    pos[0] += (dist_x + 1e-10)*omega_x
                    neut[1] = pos[0]
                    neut[2] = pos[1]
                    if (implicit_capture) and (Sig_a[cell[0], cell[1]] > 0):
                        scalar_flux_tl[cell[0],cell[1]] += neut[0]*((1.0 - 
                                                                     np.exp(-dist_x*Sig_a[cell[0], cell[1]]))
                                                                    /(Sig_a[cell[0], cell[1]] + 1.0e-14))
                        neut[0] *= np.exp(-dist_x*Sig_a[cell[0], cell[1]] )
                    else:
                        scalar_flux_tl[cell[0],cell[1]] += neut[0]*dist_x
            else:
                assert(0==1)
            
            #are we still in the problem?
            if ((pos[0] >= np.max(X)+ dx*0.5) or (pos[1] >= np.max(Y)+ dy*0.5) or 
                 ((pos[0]) < 1.0e-8) or ((pos[1]) < 1.0e-8)) :
                alive = 0
            if (neut[5] >= Lx) or (neut[5] < 0):
                alive = 0
            if (neut[6] >= Ly) or (neut[6] < 0):
                alive = 0
    return scalar_flux_coll/dx/dy, scalar_flux_tl/dx/dy

def lattice(Lengths,Dims):
    I = Dims[0]
    J = Dims[1]
    L = I*J
    Nx = Lengths[0]
    Ny = Lengths[1]
    hx,hy = np.array(Lengths)/np.array(Dims)
    
    Sigma_t = np.ones((I,J))*1
    Sigma_a = 0*Sigma_t
    Q = 0*Sigma_t
    for j in range(J):
        for i in range(I):
            x = (i+0.5)*hx
            y = (j+0.5)*hy

            if (x>=3.0) and (x<=4.0): 
                if (y>=3.0) and (y<=4.0):
                    Q[i,j] = 1.0
                if (y>=1.0) and (y<=2.0):
                    Sigma_t[i,j] = 10.0
                    Sigma_a[i,j] = 10.0
            if ( ((x>=1.0) and (x<=2.0)) or ((x>=5.0) and (x<=6.0))): 
                if ( ((y>=1.0) and (y<=2.0)) or
                    ((y>=3.0) and (y<=4.0)) or
                    ((y>=5.0) and (y<=6.0))):
                    Sigma_t[i,j] = 10.0
                    Sigma_a[i,j] = 10.0
            if ( ((x>=2.0) and (x<=3.0)) or ((x>=4.0) and (x<=5.0))): 
                if ( ((y>=2.0) and (y<=3.0)) or
                    ((y>=4.0) and (y<=5.0))):
                    Sigma_t[i,j] = 10.0
                    Sigma_a[i,j] = 10.0
    return Sigma_t, Sigma_a, Q
        
def expfiss(x):
    return 0.453*math.exp(-1.036*x)*math.sinh(math.sqrt(2.29*x))
